# mCREAM Graph Ensemble — Hyperparameter Analysis

Analyses the `consensus_dynamic_hparam` grid over:
- **noise_level**: low / medium / high
- **epochs**: 300 / 500
- **lambda_weight** (concept loss weight): 1.5 / 2.0
- **loss_type**: `per_expert` (orig) / `ensemble` (ens) / `both`

Experiment folder name encodes all params:
`graph_ensemble_consensus_dynamic_{level}_ep{epochs}_lam{lam}_{loss}`

## Sections
1. Load all results + parse hyperparameters
2. Summary table — task acc / concept acc / CCI per combo
3. Best 10 experiments ranked by task accuracy
4. Intervention curves for best 10
5. TensorBoard loss curves for best 10

In [ ]:
import pandas as pd, numpy as np, re
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

EXPERIMENTS_ROOT = Path('/home/dani00003/mCREAM/experiments')
LEVELS   = ['low', 'medium', 'high']
LEVEL_COLOR = {'low': '#3498db', 'medium': '#e67e22', 'high': '#e74c3c'}
LOSS_COLOR  = {'orig': '#8e44ad', 'ens': '#27ae60', 'both': '#e67e22'}
LOSS_MARKER = {'orig': 'o', 'ens': 's', 'both': '^'}

DATASETS = {
    'cfmnist': 'Complete_Concept_FMNIST',
    'celeba':  'CelebA',
    'cub':     'CUB',
}
MODEL_NAME = {
    'cfmnist': 'Standard_FashionMNIST',
    'celeba':  'Standard_CelebA',
    'cub':     'Standard_CUB',
}

KEY_METRICS = ['test_task_accuracy', 'test_concept_accuracy', 'CCI']

# ── Parse hyperparameters from experiment name ────────────────────────────────
SLUG2LOSS = {'orig': 'per_expert', 'ens': 'ensemble', 'both': 'both'}

def parse_hparam_name(name):
    """Extract (noise_level, epochs, lambda_weight, loss_slug) from experiment name."""
    m = re.match(
        r'graph_ensemble_consensus_dynamic_(low|medium|high)_ep(\d+)_lam(\d+_\d+)_(orig|ens|both)',
        name
    )
    if not m:
        return None
    return {
        'noise_level':   m.group(1),
        'epochs':        int(m.group(2)),
        'lambda_weight': float(m.group(3).replace('_', '.')),
        'loss_slug':     m.group(4),
        'loss_type':     SLUG2LOSS[m.group(4)],
    }

print('Setup done.')
print(f'Root exists: {EXPERIMENTS_ROOT.exists()}')

In [ ]:
def load_hparam_results(root, ds_key):
    """Load all hparam experiment CSVs for one dataset.
    Returns DataFrame with one row per seed per experiment, hyperparams parsed as columns.
    """
    ds_name = DATASETS[ds_key]
    ens_dir = root / ds_name / 'train_cbm' / 'mCREAM_GraphEnsemble'
    if not ens_dir.exists():
        print(f'[SKIP] {ens_dir} not found')
        return pd.DataFrame()

    rows = []
    for exp_dir in sorted(ens_dir.iterdir()):
        if not exp_dir.is_dir(): continue
        hp = parse_hparam_name(exp_dir.name)
        if hp is None: continue

        for seed_dir in sorted(exp_dir.glob('seed_*/lightning_logs/version_*')):
            seed = int(seed_dir.parent.parent.name.split('_')[1])
            for csv_f in sorted(seed_dir.glob('*.csv')):
                if any(x in csv_f.name for x in ['perc_','_set_','intervention','exogenous','alpha']):
                    continue
                try:
                    df = pd.read_csv(csv_f)
                    df['dataset']      = ds_key
                    df['exp_name']     = exp_dir.name
                    df['seed']         = seed
                    df['noise_level']  = hp['noise_level']
                    df['epochs']       = hp['epochs']
                    df['lambda_weight']= hp['lambda_weight']
                    df['loss_type']    = hp['loss_type']
                    df['loss_slug']    = hp['loss_slug']
                    rows.append(df)
                except: pass

    if not rows:
        print(f'No hparam results found for {ds_key}')
        return pd.DataFrame()
    df = pd.concat(rows, ignore_index=True)
    print(f'{ds_key}: {len(df)} rows | {df["exp_name"].nunique()} experiments | seeds={sorted(df["seed"].unique())}')
    return df


def load_hparam_interventions(root, ds_key):
    """Load intervention_results.csv for all hparam experiments."""
    ds_name = DATASETS[ds_key]
    ens_dir = root / ds_name / 'train_cbm' / 'mCREAM_GraphEnsemble'
    if not ens_dir.exists(): return pd.DataFrame()

    rows = []
    for exp_dir in sorted(ens_dir.iterdir()):
        if not exp_dir.is_dir(): continue
        hp = parse_hparam_name(exp_dir.name)
        if hp is None: continue
        for csv_f in exp_dir.rglob('intervention_results.csv'):
            try:
                df = pd.read_csv(csv_f)
                seed = None
                for p in csv_f.parts:
                    if p.startswith('seed_'): seed = int(p.split('_')[1])
                df['dataset']      = ds_key
                df['exp_name']     = exp_dir.name
                df['seed']         = seed
                df['noise_level']  = hp['noise_level']
                df['epochs']       = hp['epochs']
                df['lambda_weight']= hp['lambda_weight']
                df['loss_type']    = hp['loss_type']
                df['loss_slug']    = hp['loss_slug']
                rows.append(df)
            except: pass
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


# ── Load all datasets ─────────────────────────────────────────────────────────
all_dfs    = {}
all_interv = {}
for ds_key in DATASETS:
    all_dfs[ds_key]    = load_hparam_results(EXPERIMENTS_ROOT, ds_key)
    all_interv[ds_key] = load_hparam_interventions(EXPERIMENTS_ROOT, ds_key)

combined = pd.concat([df for df in all_dfs.values() if len(df)>0], ignore_index=True)
print(f'\nTotal rows: {len(combined)} | datasets: {combined["dataset"].unique() if len(combined) else []}')

---
# Section 1 — Summary Table: All Hyperparameter Combinations

In [ ]:
if len(combined) == 0:
    print('No data yet. Jobs still running or results not synced.')
else:
    GROUP_COLS = ['dataset', 'noise_level', 'epochs', 'lambda_weight', 'loss_slug']
    av = [m for m in KEY_METRICS if m in combined.columns]

    for ds_key, df in all_dfs.items():
        if len(df) == 0: continue
        av_ds = [m for m in av if m in df.columns]
        means = df.groupby(['noise_level','epochs','lambda_weight','loss_slug'])[av_ds].mean()
        stds  = df.groupby(['noise_level','epochs','lambda_weight','loss_slug'])[av_ds].std()
        sm = pd.DataFrame({
            m: means[m].map('{:.4f}'.format) + ' ± ' + stds[m].fillna(0).map('{:.4f}'.format)
            for m in av_ds
        })
        print(f'\n=== {ds_key} — all {len(sm)} hyperparameter combinations ===')
        display(sm)

---
# Section 2 — Heatmaps: Loss Type × Lambda per Noise Level

In [ ]:
if len(combined) > 0:
    av = [m for m in KEY_METRICS if m in combined.columns]
    LOSS_SLUGS = ['orig', 'ens', 'both']
    LAMBDAS    = sorted(combined['lambda_weight'].unique())

    for ds_key, df in all_dfs.items():
        if len(df) == 0: continue
        for metric in av:
            if metric not in df.columns: continue
            for ep in sorted(df['epochs'].unique()):
                sub = df[df['epochs']==ep]
                fig, axes = plt.subplots(1, len(LEVELS), figsize=(5*len(LEVELS), 4), sharey=False)
                if not hasattr(axes,'__len__'): axes=[axes]
                fig.suptitle(f'{ds_key} | {metric} | epochs={ep}\n'
                             f'rows=loss_type  cols=lambda_weight  (mean across seeds)',
                             fontsize=11, fontweight='bold')
                for ax, level in zip(axes, LEVELS):
                    lv = sub[sub['noise_level']==level]
                    if lv.empty: ax.set_title(level); ax.axis('off'); continue
                    pivot = lv.groupby(['loss_slug','lambda_weight'])[metric].mean().unstack()
                    pivot = pivot.reindex(index=[s for s in LOSS_SLUGS if s in pivot.index],
                                         columns=sorted(pivot.columns))
                    sns.heatmap(pivot, ax=ax, annot=True, fmt='.4f', cmap='YlGn',
                                linewidths=0.5, cbar=True)
                    ax.set_title(f'{level} noise', fontsize=10)
                    ax.set_xlabel('lambda_weight'); ax.set_ylabel('loss_type' if ax is axes[0] else '')
                plt.tight_layout()
                plt.savefig(f'hparam_heatmap_{ds_key}_{metric}_ep{ep}.png', dpi=150, bbox_inches='tight')
                plt.show()

---
# Section 3 — Best 10 Experiments per Dataset (ranked by task accuracy)

In [ ]:
best10_per_ds = {}  # ds_key -> list of top-10 exp_names

if len(combined) > 0:
    av = [m for m in KEY_METRICS if m in combined.columns]
    for ds_key, df in all_dfs.items():
        if len(df) == 0: continue
        if 'test_task_accuracy' not in df.columns: continue

        ranked = (
            df.groupby(['exp_name','noise_level','epochs','lambda_weight','loss_slug'])
              [av].mean()
              .reset_index()
              .sort_values('test_task_accuracy', ascending=False)
              .head(10)
        )
        best10_per_ds[ds_key] = ranked['exp_name'].tolist()

        display_cols = ['noise_level','epochs','lambda_weight','loss_slug'] + av
        display_cols = [c for c in display_cols if c in ranked.columns]
        print(f'\n=== {ds_key} — Top 10 experiments by task accuracy ===')
        display(
            ranked[display_cols]
              .reset_index(drop=True)
              .style.highlight_max(subset=[m for m in av if m in ranked.columns],
                                   color='#d4f1c0')
              .format({m: '{:.4f}' for m in av if m in ranked.columns})
        )

In [ ]:
# Bar chart of top 10 — task acc with std error bars
if len(combined) > 0:
    for ds_key, top_names in best10_per_ds.items():
        if not top_names: continue
        df = all_dfs[ds_key]
        if 'test_task_accuracy' not in df.columns: continue
        sub = df[df['exp_name'].isin(top_names)]
        agg = sub.groupby('exp_name')['test_task_accuracy'].agg(['mean','std']).reset_index()
        agg = agg.sort_values('mean', ascending=False)

        # Color bars by loss_slug
        def slug_from_name(name):
            m = re.search(r'_(orig|ens|both)$', name)
            return m.group(1) if m else 'orig'
        colors = [LOSS_COLOR[slug_from_name(n)] for n in agg['exp_name']]

        fig, ax = plt.subplots(figsize=(12, 4))
        bars = ax.barh(range(len(agg)), agg['mean'], xerr=agg['std'],
                       color=colors, alpha=0.85, capsize=4, height=0.6)
        ax.set_yticks(range(len(agg)))
        ax.set_yticklabels(
            [n.replace('graph_ensemble_consensus_dynamic_','') for n in agg['exp_name']],
            fontsize=8
        )
        ax.invert_yaxis()
        ax.set_xlabel('Task Accuracy (mean ± std across seeds)')
        ax.set_title(f'{ds_key} — Top 10 experiments', fontsize=11, fontweight='bold')
        from matplotlib.patches import Patch
        ax.legend(handles=[Patch(color=v, label=k) for k,v in LOSS_COLOR.items()],
                  title='loss_type', fontsize=8, loc='lower right')
        plt.tight_layout()
        plt.savefig(f'hparam_top10_{ds_key}.png', dpi=150, bbox_inches='tight')
        plt.show()

---
# Section 4 — Intervention Curves for Best 10 Experiments

In [ ]:
for ds_key, top_names in best10_per_ds.items():
    if not top_names: continue
    iv = all_interv.get(ds_key, pd.DataFrame())
    if len(iv) == 0:
        print(f'{ds_key}: No intervention data yet.'); continue

    iv_top = iv[iv['exp_name'].isin(top_names)]
    if len(iv_top) == 0:
        print(f'{ds_key}: Intervention data exists but not for top-10 experiments yet.'); continue

    has_group = 'group_interventions' in iv_top.columns

    # ── Individual concept interventions ──────────────────────────────────────
    iv_ind = iv_top[iv_top['group_interventions']==False] if has_group else iv_top

    # Get task accuracy ranking for color ordering
    df = all_dfs[ds_key]
    rank_map = {}
    if 'test_task_accuracy' in df.columns:
        ranked = (df[df['exp_name'].isin(top_names)]
                    .groupby('exp_name')['test_task_accuracy'].mean()
                    .sort_values(ascending=False))
        rank_map = {name: i for i, name in enumerate(ranked.index)}

    cmap = plt.cm.RdYlGn
    fig, axes = plt.subplots(1, len(LEVELS), figsize=(6*len(LEVELS), 5), sharey=True)
    if not hasattr(axes,'__len__'): axes=[axes]
    fig.suptitle(f'{ds_key} — Intervention curves: Top 10 experiments\n'
                 f'(individual concept interventions, ranked by task accuracy — green=best)',
                 fontsize=11, fontweight='bold')

    for ax, level in zip(axes, LEVELS):
        lv = iv_ind[iv_ind['noise_level']==level]
        if lv.empty: ax.set_title(f'{level} — no data'); ax.axis('off'); continue
        for exp_name in top_names:
            sub = lv[lv['exp_name']==exp_name]
            if sub.empty: continue
            agg = sub.groupby('num_interventions')['test_task_accuracy'].agg(['mean','std']).reset_index()
            rank = rank_map.get(exp_name, len(top_names)-1)
            col  = cmap(1 - rank / max(len(top_names)-1, 1))
            lw   = 3.0 if rank == 0 else 1.5
            label = exp_name.replace('graph_ensemble_consensus_dynamic_','')[:35]
            ax.plot(agg['num_interventions'], agg['mean'],
                    color=col, lw=lw, marker='o', markersize=3, label=label)
            ax.fill_between(agg['num_interventions'],
                            agg['mean']-agg['std'], agg['mean']+agg['std'],
                            alpha=0.06, color=col)
        ax.set_title(f'{level} noise', fontsize=10)
        ax.set_xlabel('Concepts replaced')
        ax.set_ylabel('Task Accuracy' if level==LEVELS[0] else '')
        ax.tick_params(labelsize=8)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='lower center', ncol=5, fontsize=6,
               bbox_to_anchor=(0.5, -0.12), framealpha=0.9)
    plt.tight_layout()
    plt.savefig(f'hparam_interv_top10_{ds_key}.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ── Group/mutex interventions (if available) ───────────────────────────────
    if has_group:
        iv_grp = iv_top[iv_top['group_interventions']==True]
        if not iv_grp.empty:
            fig, axes = plt.subplots(1, len(LEVELS), figsize=(6*len(LEVELS), 5), sharey=True)
            if not hasattr(axes,'__len__'): axes=[axes]
            fig.suptitle(f'{ds_key} — Group/mutex intervention curves: Top 10\n'
                         f'(green=best ranked by task accuracy)',
                         fontsize=11, fontweight='bold')
            for ax, level in zip(axes, LEVELS):
                lv = iv_grp[iv_grp['noise_level']==level]
                if lv.empty: ax.set_title(f'{level} — no data'); ax.axis('off'); continue
                for exp_name in top_names:
                    sub = lv[lv['exp_name']==exp_name]
                    if sub.empty: continue
                    agg = sub.groupby('num_interventions')['test_task_accuracy'].agg(['mean','std']).reset_index()
                    rank = rank_map.get(exp_name, len(top_names)-1)
                    col  = cmap(1 - rank / max(len(top_names)-1, 1))
                    ax.plot(agg['num_interventions'], agg['mean'],
                            color=col, lw=3.0 if rank==0 else 1.5,
                            marker='s', markersize=4, ls='--')
                ax.set_title(f'{level} noise', fontsize=10)
                ax.set_xlabel('Mutex groups replaced')
                ax.set_ylabel('Task Accuracy' if level==LEVELS[0] else '')
                ax.tick_params(labelsize=8)
            plt.tight_layout()
            plt.savefig(f'hparam_interv_group_top10_{ds_key}.png', dpi=150, bbox_inches='tight')
            plt.show()

---
# Section 5 — TensorBoard Loss Curves for Best 10 Experiments

In [ ]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

TB_TAGS = ['train_task_loss', 'train_concept_loss', 'train_total_loss', 'val_total_loss']
TB_COLOR = {
    'train_task_loss':    '#e74c3c',
    'train_concept_loss': '#3498db',
    'train_total_loss':   '#2c3e50',
    'val_total_loss':     '#8e44ad',
}

def load_tb_for_exp(root, ds_key, exp_name, tags=TB_TAGS):
    """Load TensorBoard scalars for all seeds of one hparam experiment.
    Returns dict: seed -> {tag -> DataFrame(step, value)}
    """
    ds_name = DATASETS[ds_key]
    exp_dir = root / ds_name / 'train_cbm' / 'mCREAM_GraphEnsemble' / exp_name
    if not exp_dir.exists(): return {}
    result = {}
    for seed_dir in sorted(exp_dir.glob('seed_*')):
        seed = int(seed_dir.name.split('_')[1])
        for ver_dir in sorted(seed_dir.glob('lightning_logs/version_*')):
            for ef in sorted(ver_dir.glob('events.out.tfevents.*')):
                try:
                    ea = EventAccumulator(str(ef), size_guidance={'scalars': 0})
                    ea.Reload()
                    avail = ea.Tags()['scalars']
                    data = {}
                    for tag in tags:
                        if tag in avail:
                            evs = ea.Scalars(tag)
                            data[tag] = pd.DataFrame({'step': [e.step for e in evs],
                                                      'value': [e.value for e in evs]})
                    if any('train' in k for k in data):
                        result[seed] = data; break
                except: pass
    return result


def plot_tb_for_exp(tb_data, exp_name, ax_dict):
    """Plot mean±std loss curves into provided axes dict {tag: ax}."""
    for tag, ax in ax_dict.items():
        col = TB_COLOR.get(tag, '#555')
        seed_dfs = []
        for seed, data in tb_data.items():
            if tag not in data: continue
            df = data[tag].set_index('step').rename(columns={'value': seed})
            seed_dfs.append(df)
            ax.plot(df.index, df[seed], color=col, lw=0.6, alpha=0.25)
        if seed_dfs:
            combined = pd.concat(seed_dfs, axis=1)
            mean = combined.mean(axis=1); std = combined.std(axis=1)
            ax.plot(mean.index, mean.values, color=col, lw=2.5)
            ax.fill_between(mean.index, mean-std, mean+std, alpha=0.15, color=col)
        ax.set_title(tag.replace('train_','').replace('_loss','').replace('_',' '), fontsize=8)
        ax.set_xlabel('Step'); ax.tick_params(labelsize=7)

print('TensorBoard helpers ready.')

In [ ]:
# ── For each dataset: load TB for top-10 and plot a grid ─────────────────────
# Layout: 10 rows (experiments) × 4 cols (loss tags)
for ds_key, top_names in best10_per_ds.items():
    if not top_names: continue

    tags_to_plot = TB_TAGS
    n_exp = len(top_names)
    fig, axes = plt.subplots(n_exp, len(tags_to_plot),
                             figsize=(4*len(tags_to_plot), 3*n_exp),
                             sharey='col')
    if n_exp == 1: axes = [axes]
    fig.suptitle(f'{ds_key} — TensorBoard loss curves: Top {n_exp} experiments\n'
                 f'(rows = experiments ranked best→worst, cols = loss components)',
                 fontsize=11, fontweight='bold')

    loaded = 0
    for row_idx, exp_name in enumerate(top_names):
        row_axes = axes[row_idx] if hasattr(axes[row_idx], '__len__') else [axes[row_idx]]
        ax_dict  = {tag: ax for tag, ax in zip(tags_to_plot, row_axes)}

        tb_data = load_tb_for_exp(EXPERIMENTS_ROOT, ds_key, exp_name)
        short   = exp_name.replace('graph_ensemble_consensus_dynamic_', '')

        if not tb_data:
            for ax in row_axes:
                ax.text(0.5, 0.5, 'no TB data', ha='center', va='center',
                        transform=ax.transAxes, fontsize=8, color='gray')
            row_axes[0].set_ylabel(f'#{row_idx+1}\n{short}', fontsize=6, rotation=0,
                                    ha='right', labelpad=60)
            continue

        plot_tb_for_exp(tb_data, exp_name, ax_dict)
        row_axes[0].set_ylabel(f'#{row_idx+1}\n{short}', fontsize=6, rotation=0,
                                ha='right', labelpad=60)
        loaded += 1

    # Column headers on top row
    for ax, tag in zip(axes[0] if hasattr(axes[0],'__len__') else [axes[0]], tags_to_plot):
        ax.set_title(tag.replace('train_','').replace('_loss','').replace('_',' '),
                     fontsize=9, fontweight='bold')

    print(f'{ds_key}: loaded TB for {loaded}/{n_exp} experiments')
    plt.tight_layout()
    plt.savefig(f'hparam_tb_top10_{ds_key}.png', dpi=120, bbox_inches='tight')
    plt.show()

In [ ]:
# ── Alternative view: overlay top 10 on single axes per loss tag ─────────────
# Each experiment is one mean line, colored by loss_slug
for ds_key, top_names in best10_per_ds.items():
    if not top_names: continue

    fig, axes = plt.subplots(1, len(TB_TAGS), figsize=(5*len(TB_TAGS), 4), sharey=False)
    if not hasattr(axes,'__len__'): axes=[axes]
    fig.suptitle(f'{ds_key} — Loss overlay: Top 10 experiments\n'
                 f'colour = loss_type   thickness ∝ rank (thicker = better)',
                 fontsize=11, fontweight='bold')

    df_rank = all_dfs[ds_key]
    if 'test_task_accuracy' in df_rank.columns:
        ranked_acc = (df_rank[df_rank['exp_name'].isin(top_names)]
                        .groupby('exp_name')['test_task_accuracy'].mean()
                        .sort_values(ascending=False))
    else:
        ranked_acc = pd.Series(index=top_names, data=range(len(top_names), 0, -1))

    for rank, exp_name in enumerate(ranked_acc.index):
        tb_data = load_tb_for_exp(EXPERIMENTS_ROOT, ds_key, exp_name)
        if not tb_data: continue
        slug  = re.search(r'_(orig|ens|both)$', exp_name)
        col   = LOSS_COLOR[slug.group(1)] if slug else '#555'
        lw    = max(0.8, 3.0 - rank * 0.25)
        alpha = max(0.3, 1.0 - rank * 0.08)
        short = exp_name.replace('graph_ensemble_consensus_dynamic_','')

        for ax, tag in zip(axes, TB_TAGS):
            seed_dfs = []
            for seed, data in tb_data.items():
                if tag not in data: continue
                seed_dfs.append(data[tag].set_index('step').rename(columns={'value': seed}))
            if not seed_dfs: continue
            mean = pd.concat(seed_dfs, axis=1).mean(axis=1)
            ax.plot(mean.index, mean.values, color=col, lw=lw, alpha=alpha,
                    label=f'#{rank+1} {short[:30]}' if rank < 5 else '')

    for ax, tag in zip(axes, TB_TAGS):
        ax.set_title(tag.replace('train_','').replace('_loss','').replace('_',' '), fontsize=9)
        ax.set_xlabel('Step'); ax.tick_params(labelsize=8)

    axes[0].set_ylabel('Loss')
    from matplotlib.patches import Patch
    axes[-1].legend(
        handles=[Patch(color=v, label=k) for k,v in LOSS_COLOR.items()],
        title='loss_type', fontsize=8, loc='upper right'
    )
    plt.tight_layout()
    plt.savefig(f'hparam_tb_overlay_{ds_key}.png', dpi=150, bbox_inches='tight')
    plt.show()

---
# Section 6 — Factor Analysis: Which Hyperparameter Matters Most?

In [ ]:
# Marginal effect of each hyperparameter averaged across all others
if len(combined) > 0 and 'test_task_accuracy' in combined.columns:
    FACTORS = ['noise_level', 'epochs', 'lambda_weight', 'loss_slug']

    for ds_key, df in all_dfs.items():
        if len(df) == 0 or 'test_task_accuracy' not in df.columns: continue

        fig, axes = plt.subplots(1, len(FACTORS), figsize=(4*len(FACTORS), 4), sharey=False)
        fig.suptitle(f'{ds_key} — Marginal effect of each hyperparameter on task accuracy\n'
                     f'(mean ± std across all other factor combinations)',
                     fontsize=11, fontweight='bold')

        for ax, factor in zip(axes, FACTORS):
            agg = df.groupby(factor)['test_task_accuracy'].agg(['mean','std']).reset_index()
            colors = {
                'noise_level':   [LEVEL_COLOR.get(str(v), '#888') for v in agg[factor]],
                'loss_slug':     [LOSS_COLOR.get(str(v), '#888') for v in agg[factor]],
                'epochs':        ['#2980b9'] * len(agg),
                'lambda_weight': ['#e67e22'] * len(agg),
            }[factor]
            ax.bar(range(len(agg)), agg['mean'], yerr=agg['std'],
                   color=colors, alpha=0.8, capsize=5)
            ax.set_xticks(range(len(agg)))
            ax.set_xticklabels([str(v) for v in agg[factor]], fontsize=9)
            ax.set_title(factor, fontsize=10)
            ax.set_ylabel('Task Accuracy' if ax is axes[0] else '')
            ax.tick_params(labelsize=8)

        plt.tight_layout()
        plt.savefig(f'hparam_factors_{ds_key}.png', dpi=150, bbox_inches='tight')
        plt.show()